### Loading data in hierarchal HDF5 format

Subject -> Stimulus Modality -> Stimulus Font -> Parity or Control

In [32]:
import os
import pandas as pd
import mne
from pr_fe import FeatureExtractor
import h5py
import numpy as np

In [33]:
input_list_path = 'data/mat_files_cleaned.txt'
data_dir = 'data'
output_base_dir = 'h5_sep1'

fe = FeatureExtractor()

In [ ]:
def parse_filename(filename):
    base = os.path.basename(filename).replace('.mat', '')
    parts = base.split('_')
    
    subject = next((p[1:] for p in reversed(parts) if p.startswith('S')), 'Unknown')
    
    try:
        idx = parts.index('epbin') + 1
    except ValueError:
        idx = 1

    category = parts[idx] if len(parts) > idx else None
    font_type = parts[idx+1] if len(parts) > idx+1 else None
    condition_code = parts[idx+3] if len(parts) > idx+3 else None
    
    is_20f = font_type == '20F' and len(parts) > idx+2 and parts[idx+2] in ['A', 'S']
    specific_font = parts[idx+2] if is_20f else None
    
    return subject, category, font_type, specific_font, condition_code

In [35]:
def load_and_prepare_csv(filepath):
    df = pd.read_csv(filepath)
    
    df_cond0 = df[df['condition'] == 0].pivot(index='time', columns='channel', values='value')
    df_cond1 = df[df['condition'] == 1].pivot(index='time', columns='channel', values='value')
    
    df_cond0.columns = df_cond0.columns.astype(str)
    df_cond1.columns = df_cond1.columns.astype(str)
    common_channels = sorted(list(set(df_cond0.columns) & set(df_cond1.columns)))
    df_cond0 = df_cond0[common_channels]
    df_cond1 = df_cond1[common_channels]
    
    return df_cond0, df_cond1, common_channels

In [36]:
def process_condition(df_signal, channels, condition_label):
    ch_types = ['eeg'] * len(channels)
    sfreq = fe.sampling_rate
    data = df_signal[channels].T.values 

    info = mne.create_info(ch_names=channels, sfreq=sfreq, ch_types=ch_types)
    raw = mne.io.RawArray(data, info)

    features_df = fe.merging_feature_data(raw, df_signal)
    
    features_df = features_df.apply(pd.to_numeric, errors='coerce')
    features_df = features_df.select_dtypes(include=[np.number])
    features_df = features_df.dropna(axis=1, how='all')
    
    return features_df

In [ ]:
with open(input_list_path, 'r') as f:
    files = [line.strip() for line in f if line.strip()]

for file_rel_path in files:
    csv_rel_path = file_rel_path.replace('.mat', '.csv')
    file_path = os.path.join(data_dir, csv_rel_path)

    if not os.path.isfile(file_path):
        print(f"File not found: {file_path}, skipping.")
        continue

    print(f"Processing {file_path}...")

    subject, category, font_type, specific_font, condition_code = parse_filename(file_rel_path)
    
    if None in [subject, category, font_type]:
        print(f"Skipping {file_path} - missing essential info")
        continue
        
    condition_types = []
    if condition_code and 'Par' in condition_code:
        condition_types.append('Parity')
    if condition_code and 'C1' in condition_code:
        condition_types.append('Control')
    if not condition_types:  
        condition_types = ['Parity', 'Control']
        print(f"Processing both conditions for {file_path}")
    
    df_cond0, df_cond1, common_channels = load_and_prepare_csv(file_path)

    dir_parts = [output_base_dir, f'S{subject}', category, font_type]
    if specific_font:
        dir_parts.append(specific_font)
    
    output_dir = os.path.join(*dir_parts)
    os.makedirs(output_dir, exist_ok=True)
    
    for condition_type in condition_types:
        for cond_num, df in [('0', df_cond0), ('1', df_cond1)]:
            try:
                features_df = process_condition(df, common_channels, cond_num)
                
                output_file = os.path.join(output_dir, f'{condition_type}_{cond_num}.h5')
                group_name = '/'.join(dir_parts[1:])  

                with h5py.File(output_file, 'a') as hdf5_file:
                    group = hdf5_file.require_group(group_name)
                    
                    dataset_name = f'{condition_type}_{cond_num}'
                    if dataset_name in group:
                        print(f"Dataset {dataset_name} exists, skipping.")
                        continue

                    group.create_dataset(dataset_name, 
                                      data=features_df.to_numpy(), 
                                      chunks=True)
                    group.attrs['columns'] = np.array(features_df.columns, dtype='S')
                    group.attrs['condition_type'] = condition_type
                    group.attrs['condition_num'] = cond_num

                print(f"Saved {condition_type}_{cond_num} to {output_file}")
                
            except Exception as e:
                print(f"Failed to process {condition_type}_{cond_num}: {str(e)}")

Processing data/epbin_Dig_1F_C1_21_rr_fixAF7_fixF7_icfilt_ica_ep1_but_chanlocs_chansel_chanlabels_S19.csv...
Processing both conditions for data/epbin_Dig_1F_C1_21_rr_fixAF7_fixF7_icfilt_ica_ep1_but_chanlocs_chansel_chanlabels_S19.csv
Creating RawArray with float64 data, n_channels=70, n_times=30720
    Range : 0 ... 30719 =      0.000 ...    59.998 secs
Ready.
Effective window size : 0.500 (s)
Saved Parity_0 to h5_sep1/S19/Dig/1F/Parity_0.h5
Creating RawArray with float64 data, n_channels=70, n_times=30720
    Range : 0 ... 30719 =      0.000 ...    59.998 secs
Ready.
Effective window size : 0.500 (s)
Saved Parity_1 to h5_sep1/S19/Dig/1F/Parity_1.h5
Creating RawArray with float64 data, n_channels=70, n_times=30720
    Range : 0 ... 30719 =      0.000 ...    59.998 secs
Ready.
Effective window size : 0.500 (s)
Saved Control_0 to h5_sep1/S19/Dig/1F/Control_0.h5
Creating RawArray with float64 data, n_channels=70, n_times=30720
    Range : 0 ... 30719 =      0.000 ...    59.998 secs
Ready